# The canonical coordinates of the coherent coupler

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. We show that it is possible to transform the coherent coupler to a canonical coordinate system in which the solutions are single valued functions, namely Kronecker theta functions. In doing so, we show that the classic trick of turning the quartic to a cubic in the elliptic differential equation leaves the differential system of coupled modes invariant in terms of the type of terms that appear but it delicately balances coefficients to kill the quartic term. 

In [1]:
from sympy import *

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi) = symbols(
    '''delta, nu, Aeff, chi'''
)

gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')

kappa = Function('kappa')
phi = Function('phi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lam = IndexedBase('lambda')
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

## Elliptic function identities

In [2]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)



# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(\wp'(x, g2*c**4, g3*c**6), \wp'(x*c, g2, g3)*c**3)

Eq(omega[1], f(1, g2, g3))

Eq(omega[1]*c, f(1, g2/c**4, g3/c**6))

Eq(omega[3], f(3, g2, g3))

Eq(omega[3]*c, f(3, g2/c**4, g3/c**6))

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [3]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1
sum_r1j_1
r1j_log_part

for _ in Lams:
    _

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

### The original parameters

In [4]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [5]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

drhop_b
drhop_d
drho_2z
drho_2z_b

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j - 1)*d[j]/2, (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) - 4*rho(z) + 2*gamma[1] + 2*gamma[2])

### Reorganising the original equations of motion

In [6]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'

uv_j_rho

uprodj
vprodj
a_0_eq

duj
dvj
duvj

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(Product(v(z, mu[j]), (j, 1, 2)), Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[k]), (k, 1, 2)) + Product(v(z, mu[k]), (k, 1, 2)))

## Gauge transform to the hat coordinates

In this section we implement a gauge transform to remove XPM, i.e., to remove the log of $\sigma$ terms from the exponential:

In [7]:
uU_sub = Eq(
    u(z,mu[j]), 
    uhat(z, mu[j])
    *exp(-phi(z,j))
)
vV_sub = Eq(
    v(z,mu[j]), 
    vhat(z, mu[j])
    *exp(phi(z,j))
)
Uu_sub = Eq(uU_sub.rhs*exp(phi(z, j)), uU_sub.lhs*exp(phi(z, j)))
Vv_sub = Eq(vV_sub.rhs*exp(-phi(z, j)), vV_sub.lhs*exp(-phi(z, j)))

uU_sub
vV_sub

Uu_sub
Vv_sub

Eq(u(z, mu[j]), uhat(z, mu[j])*exp(-phi(z, j)))

Eq(v(z, mu[j]), vhat(z, mu[j])*exp(phi(z, j)))

Eq(uhat(z, mu[j]), u(z, mu[j])*exp(phi(z, j)))

Eq(vhat(z, mu[j]), v(z, mu[j])*exp(-phi(z, j)))

In [8]:
(-r[1, 1]+r[1,2])/2 + (r[1, 1]+r[1,2])/2

r[1, 2]

In [9]:
r[1,1] + (-r[1, 1]+r[1,2])/2 + (r[1, 1]+r[1,2])/2
r[1,2] - (-r[1, 1]+r[1,2])/2 - (r[1, 1]+r[1,2])/2

r[1, 1] + r[1, 2]

0

In [10]:
phi_opt_1 = log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*(-r[1, 1]+r[1,2])/2
phi_opt_2 = log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*(-r[1, 1]+r[1,2])/2

phi_j_eqs = [
    Eq(phi(z,1), phi_opt_1),
    Eq(phi(z,2), -phi_opt_1)
]
phi_j_subs = [_.args for _ in phi_j_eqs]

gauge_phi_sum = Eq(
    Sum(phi(z,j),(j,1,2)), 
    Sum(phi(z,j),(j,1,2)).doit().subs(phi_j_subs)
)

uhat_gauge_1 = Eq(
    Uu_sub.lhs.subs(j,1),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_1 = Eq(
    Vv_sub.lhs.subs(j,1),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,1),
        phi_j_subs[0]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

uhat_gauge_2 = Eq(
    Uu_sub.lhs.subs(j,2),
    Uu_sub.rhs.subs([
        u_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

vhat_gauge_2 = Eq(
    Vv_sub.lhs.subs(j,2),
    Vv_sub.rhs.subs([
        v_sqrt_wp_z0_z1.args,
        (j,2),
        phi_j_subs[1]
    ]).expand().simplify().subs([
        (r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0]),
        sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
    ]).simplify()
)

uhat_gauge_1
vhat_gauge_1
uhat_gauge_2
vhat_gauge_2

for _ in phi_j_eqs:
    _
    
gauge_phi_sum

Eq(uhat(z, mu[1]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*r[0, 1] + epsilon[1])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[1], g2, g3)*exp(-z*r[0, 1] - epsilon[1])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[1], g2, g3)*d[4]**(1/4)))

Eq(uhat(z, mu[2]), sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[2], g2, g3)*exp(z*r[0, 2] + epsilon[2])/(sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[2], g2, g3)*exp(-z*r[0, 2] - epsilon[2])/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[2], g2, g3)*d[4]**(1/4)))

Eq(phi(z, 1), (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(phi(z, 2), -(-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

Eq(Sum(phi(z, j), (j, 1, 2)), 0)

In [11]:
# upsilon captures the sign ambiguity

sig_sqrt_1 = Eq(
    sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/
    sqrt(
        sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
        (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
    ),
    upsilon*sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 - z1, g2, g3)
)

sig_sqrt_2 = Eq(
    1/(
        sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*
        sqrt(
            sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/
            (sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)
        )
    ),
    sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/sigma(z - z0 + z1, g2, g3)/upsilon
)
assert (sig_sqrt_1.lhs*sig_sqrt_2.lhs/(sig_sqrt_1.rhs*sig_sqrt_2.rhs)) == 1, 'sqrt ratios not 1'
sig_sqrt_subs = [
    sig_sqrt_1.args,
    sig_sqrt_2.args
]

sig_sqrt_1
sig_sqrt_2

Eq(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2)), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)*upsilon/sigma(z - z0 - z1, g2, g3))

Eq(1/(sqrt(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)/(sigma(z1, g2, g3)**2*sigma(z - z0, g2, g3)**2))), sigma(z1, g2, g3)*sigma(z - z0, g2, g3)/(sigma(z - z0 + z1, g2, g3)*upsilon))

In [12]:
uhat_gauge_1.subs(sig_sqrt_subs)
vhat_gauge_1.subs(sig_sqrt_subs)
uhat_gauge_2.subs(sig_sqrt_subs)
vhat_gauge_2.subs(sig_sqrt_subs)

Eq(uhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[1], g2, g3)*exp(z*r[0, 1] + epsilon[1])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[1]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[1], g2, g3)*exp(-z*r[0, 1] - epsilon[1])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*sigma(-z0 + mu[1], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

Eq(uhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[2], g2, g3)*exp(z*r[0, 2] + epsilon[2])*upsilon/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 - z1, g2, g3)*d[4]**(1/4)))

Eq(vhat(z, mu[2]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[2], g2, g3)*exp(-z*r[0, 2] - epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*sigma(-z0 + mu[2], g2, g3)*sigma(z - z0 + z1, g2, g3)*d[4]**(1/4)*upsilon))

In [13]:
uv_gamma_cons =Eq(
    uv_j_rho.lhs - uv_j_rho.lhs.subs(j,k),
    uv_j_rho.rhs - uv_j_rho.rhs.subs(j,k)
)

uv_gamma_cons_gauge = Eq(
    uv_gamma_cons.lhs.subs([
        uU_sub.args, 
        vV_sub.args,
        uU_sub.subs(j,k).args, 
        vV_sub.subs(j,k).args
    ]).simplify(),
    uv_gamma_cons.rhs
)
uv_gam_gauge_k_to_j = Eq(
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.lhs,
    uhat(z, mu[j])*vhat(z, mu[j]) - uv_gamma_cons_gauge.rhs
)


uv_j_rho
uv_gamma_cons
uv_gamma_cons_gauge

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]), gamma[j] - gamma[k])

Eq(uhat(z, mu[j])*vhat(z, mu[j]) - uhat(z, mu[k])*vhat(z, mu[k]), gamma[j] - gamma[k])

In [14]:
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) +
    log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/sqrt(d[4]) + C
)

rho_integral

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) + log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/sqrt(d[4]))

In [15]:
phi_j_eqs[0]

Eq(phi(z, 1), (-r[1, 1] + r[1, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/2)

In [44]:
r1j_a_d4 = r1j_log_part.subs(*Lams[1].replace(j,l).replace(m,j).args)

r11_min_r22 = Eq(
    r1j_a_d4.lhs.subs(j,1).doit() - r1j_a_d4.lhs.subs(j,2).doit(),
    (r1j_a_d4.rhs.subs(j,1).doit() - r1j_a_d4.rhs.subs(j,2).doit())
    .simplify().subs(ajk_syms)
)

r1j_a_d4
r11_min_r22

phi_j_eqs[0].subs([r11_min_r22.args])

Eq(r[1, j], (Sum(a[j, k], (k, 1, 2)) - Sum(a[l, k]/4, (l, 1, 2), (k, 1, 2)))/sqrt(d[4]))

Eq(r[1, 1] - r[1, 2], (a[1, 1] - a[2, 2])/sqrt(d[4]))

Eq(phi(z, 1), -(a[1, 1] - a[2, 2])*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/(2*sqrt(d[4])))

In [47]:
rho_integral

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) + log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/sqrt(d[4]))

In [72]:
gauge_subs_phi1 = [
    uU_sub.subs(j,1).args,
    vV_sub.subs(j,1).args,
    uU_sub.subs(j,2).args,
    vV_sub.subs(j,2).args,
#     (phi(z,2), - phi(z,1))
]

duhat_1_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(duj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 1))[0]
        - vhat(z, mu[2])
    ).factor()
    + vhat(z, mu[2])
    
)
duhat_2_phi = Eq(
    diff(uhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(duj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(uhat(z, mu[j]), z).subs(j, 2))[0]
        - vhat(z, mu[1])
    ).factor()
    + vhat(z, mu[1])
    
)
dvhat_1_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 1), 
    (
        solve(dvj.subs(j, 1).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 1))[0]
        + uhat(z, mu[2])
    ).factor()
    - uhat(z, mu[2])
    
)
dvhat_2_phi = Eq(
    diff(vhat(z, mu[j]), z).subs(j, 2), 
    (
        solve(dvj.subs(j, 2).doit().subs(gauge_subs_phi1).doit(), diff(vhat(z, mu[j]), z).subs(j, 2))[0]
        + uhat(z, mu[1])
    ).factor()
    - uhat(z, mu[1])
    
)
du_dv_hat_eqs = [
    duhat_1_phi,
    duhat_2_phi,
    dvhat_1_phi,
    dvhat_2_phi
]

for _ in du_dv_hat_eqs:
    _


Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*a[1, 1] - uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[1])*Derivative(phi(z, 1), z) - uhat(z, mu[1])*a[1] + vhat(z, mu[2])*exp(phi(z, 1))*exp(phi(z, 2)))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*a[2, 1] - uhat(z, mu[2])**2*vhat(z, mu[2])*a[2, 2] + uhat(z, mu[2])*Derivative(phi(z, 2), z) - uhat(z, mu[2])*a[2] + vhat(z, mu[1])*exp(phi(z, 1))*exp(phi(z, 2)))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*a[1, 1] + uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])*a[1, 2] - uhat(z, mu[2])*exp(-phi(z, 1))*exp(-phi(z, 2)) - vhat(z, mu[1])*Derivative(phi(z, 1), z) + vhat(z, mu[1])*a[1])

Eq(Derivative(vhat(z, mu[2]), z), uhat(z, mu[1])*vhat(z, mu[1])*vhat(z, mu[2])*a[2, 1] - uhat(z, mu[1])*exp(-phi(z, 1))*exp(-phi(z, 2)) + uhat(z, mu[2])*vhat(z, mu[2])**2*a[2, 2] - vhat(z, mu[2])*Derivative(phi(z, 2), z) + vhat(z, mu[2])*a[2])

In [24]:
phiz1_int_uv_hat = Eq(
    phi(z,1), 
    Integral(c[1]*uhat(z, mu[1])*vhat(z, mu[1]) + c[2]*uhat(z, mu[2])*vhat(z, mu[2]), z)
#     .subs([
#     #     (c[1], -(a[1,1] - a[2,2])/4),
#     #     (c[2], -(a[1,1] - a[2,2])/4)
#     #     (c[1], -a[1,2]),
#     #     (c[2], a[1,2])
#         (c[1], -a[1,1]/4 + a[2,2]/4 - a[1,2]),
#         (c[2], -a[1,1]/4 + a[2,2]/4 + a[1,2])
#     #     (c[1], a[2,2]/2 - a[1,2]),
#     #     (c[2], -a[1,1]/2 + a[1,2])
#     ])
)

for _ in du_dv_hat_eqs:
    _.subs([phiz1_int_uv_hat.args]).doit().expand().simplify().subs(ajk_syms)

Eq(Derivative(uhat(z, mu[1]), z), -uhat(z, mu[1])**2*vhat(z, mu[1])*a[1, 1] + uhat(z, mu[1])**2*vhat(z, mu[1])*c[1] - uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2])*c[2] - uhat(z, mu[1])*a[1] + vhat(z, mu[2]))

Eq(Derivative(uhat(z, mu[2]), z), -uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*a[1, 2] - uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*c[1] - uhat(z, mu[2])**2*vhat(z, mu[2])*a[2, 2] - uhat(z, mu[2])**2*vhat(z, mu[2])*c[2] - uhat(z, mu[2])*a[2] + vhat(z, mu[1]))

Eq(Derivative(vhat(z, mu[1]), z), uhat(z, mu[1])*vhat(z, mu[1])**2*a[1, 1] - uhat(z, mu[1])*vhat(z, mu[1])**2*c[1] + uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])*a[1, 2] - uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])*c[2] - uhat(z, mu[2]) + vhat(z, mu[1])*a[1])

Eq(Derivative(vhat(z, mu[2]), z), uhat(z, mu[1])*vhat(z, mu[1])*vhat(z, mu[2])*a[1, 2] + uhat(z, mu[1])*vhat(z, mu[1])*vhat(z, mu[2])*c[1] - uhat(z, mu[1]) + uhat(z, mu[2])*vhat(z, mu[2])**2*a[2, 2] + uhat(z, mu[2])*vhat(z, mu[2])**2*c[2] + vhat(z, mu[2])*a[2])

In [45]:
uvhat_mode_powers = [uhat(z, mu[1])*vhat(z, mu[1]), uhat(z, mu[2])*vhat(z, mu[2])]
_x1 = ((
    du_dv_hat_eqs[0].rhs.subs([phiz1_int_uv_hat.args]).doit().expand().simplify().subs(ajk_syms) 
    + a[1]*uhat(z, mu[1]) - vhat(z, mu[2])
).subs([
    ( uhat(z, mu[2])*vhat(z, mu[2]), -gamma[1] + gamma[2] + uhat(z, mu[1])*vhat(z, mu[1]))
]).factor()/uhat(z, mu[1])).collect(uvhat_mode_powers, factor)

_x2 = ((
    du_dv_hat_eqs[1].rhs.subs([phiz1_int_uv_hat.args]).doit().expand().simplify().subs(ajk_syms) 
    + a[2]*uhat(z, mu[2]) - vhat(z, mu[1])
).subs([
    ( uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - gamma[2] + uhat(z, mu[2])*vhat(z, mu[2]))
]).factor()/uhat(z, mu[2])).collect(uvhat_mode_powers, factor)

_x1
_x2

(a[1, 2] - c[2])*(gamma[1] - gamma[2]) + (-a[1, 1] - a[1, 2] + c[1] + c[2])*uhat(z, mu[1])*vhat(z, mu[1])

-(a[1, 2] + c[1])*(gamma[1] - gamma[2]) + (-a[1, 2] - a[2, 2] - c[1] - c[2])*uhat(z, mu[2])*vhat(z, mu[2])

In [46]:
_c2 = Eq(c[2], solve(Eq(-a[1,1] + c[1], - a[2,2] - c[2]), c[2])[0])


_x1.subs([_c2.args])
_x2.subs([_c2.args])

_c_subs = [
#         (c[1], -(a[1,1] - a[2,2])/4),
#         (c[2], -(a[1,1] - a[2,2])/4)
    #     (c[1], -a[1,2]),
    #     (c[2], a[1,2])
        (c[1], -a[1,1]/4 + a[2,2]/4 - a[1,2]),
        (c[2], -a[1,1]/4 + a[2,2]/4 + a[1,2])
    #     (c[1], a[2,2]/2 - a[1,2]),
    #     (c[2], -a[1,1]/2 + a[1,2])
]
_x1.subs(_c_subs)
_x2.subs(_c_subs)

(-a[1, 2] - a[2, 2])*uhat(z, mu[1])*vhat(z, mu[1]) + (gamma[1] - gamma[2])*(-a[1, 1] + a[1, 2] + a[2, 2] + c[1])

(-a[1, 1] - a[1, 2])*uhat(z, mu[2])*vhat(z, mu[2]) - (a[1, 2] + c[1])*(gamma[1] - gamma[2])

(a[1, 1]/4 - a[2, 2]/4)*(gamma[1] - gamma[2]) + (-3*a[1, 1]/2 - a[1, 2] + a[2, 2]/2)*uhat(z, mu[1])*vhat(z, mu[1])

-(-a[1, 1]/4 + a[2, 2]/4)*(gamma[1] - gamma[2]) + (a[1, 1]/2 - a[1, 2] - 3*a[2, 2]/2)*uhat(z, mu[2])*vhat(z, mu[2])

In [20]:
_phi = - r[1,1]/2 + r[1,2]/2

r[1,1] + _phi

r[1,2] - _phi

r[1, 1]/2 + r[1, 2]/2

r[1, 1]/2 + r[1, 2]/2

In [21]:
(Lams[1].subs(m,1).doit().rhs + Lams[1].subs(m,2).doit().rhs).subs(ajk_syms)

a[1, 1]/2 + a[1, 2] + a[2, 2]/2

In [51]:
duz_unified = Eq(
    Derivative(u(z, mu[j]), z)/u(z, mu[j]),
    (rhop(z) - rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z)
)
dvz_unified = Eq(
    Derivative(v(z, mu[j]), z)/v(z, mu[j]),
    (rhop(z) + rhop(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z)
)

dphi_m_Q = Eq(
    Derivative(phi(z, mu[m]), z),
    (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) 
    - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
    + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
)

dphi_m_Q_Lam = Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])

Lam0_sum_b = Eq(
    Sum(Lamda[0, m], (m, 1, 2)),
    Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2))
)
Lam1_sum_b = Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

duz_unified
dvz_unified
dphi_m_Q
dphi_m_Q_Lam
Lam0_sum_b
Lam1_sum_b

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (\rho'(z) - \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z))

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (\rho'(z) + \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z))

Eq(Derivative(phi(z, mu[m]), z), (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])

Eq(Sum(Lamda[0, m], (m, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

In [53]:
dphi_m_Q.subs( rho(z), gamma[m] - u(z, mu[m])*v(z, mu[m]))

Eq(Derivative(phi(z, mu[m]), z), (-u(z, mu[m])*v(z, mu[m]) + gamma[m])*(Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))) - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

In [60]:
gamms = solve([
    Eq(gamma[1] - gamma[2], u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2])),
    Eq(gamma[1] + gamma[2], 0)
],[gamma[1], gamma[2]])

_phi_dz =(dphi_m_Q.
 subs( rho(z), gamma[m] - u(z, mu[m])*v(z, mu[m]))
 .simplify().expand().simplify()
 .subs(ajk_syms)
)

_phi_dz

Eq(Derivative(phi(z, mu[m]), z), u(z, mu[m])*v(z, mu[m])*a[1, 1]/4 + u(z, mu[m])*v(z, mu[m])*a[1, 2]/2 + u(z, mu[m])*v(z, mu[m])*a[2, 2]/4 - u(z, mu[m])*v(z, mu[m])*a[m, 1] - u(z, mu[m])*v(z, mu[m])*a[m, 2] + a[1, 1]*gamma[1]/2 - a[1, 1]*gamma[m]/2 - a[1, 2]*gamma[m] + a[1]/2 + a[2, 2]*gamma[2]/2 - a[2, 2]*gamma[m]/2 + a[2]/2 - a[m, 1]*gamma[1] + a[m, 1]*gamma[m] - a[m, 2]*gamma[2] + a[m, 2]*gamma[m] - a[m])

In [80]:
phi_mu_eqs = [
    _phi_dz.subs(m,1).subs(gamms).simplify().subs(ajk_syms),
    _phi_dz.subs(m,2).subs(gamms).simplify().subs(ajk_syms)
]

phi_mu_eqs[0] = Eq(phi(z, mu[1]), Integral(phi_mu_eqs[0].rhs,z))
phi_mu_eqs[1] = Eq(phi(z, mu[2]), Integral(phi_mu_eqs[1].rhs,z))

phi_mu_subs = [_.args for _ in phi_mu_eqs]
for _ in phi_mu_eqs:
    _

Eq(phi(z, mu[1]), Integral(-3*u(z, mu[1])*v(z, mu[1])*a[1, 1]/4 - u(z, mu[1])*v(z, mu[1])*a[2, 2]/4 - u(z, mu[2])*v(z, mu[2])*a[1, 2]/2 + u(z, mu[2])*v(z, mu[2])*a[2, 2]/2 - a[1]/2 + a[2]/2, z))

Eq(phi(z, mu[2]), Integral(u(z, mu[1])*v(z, mu[1])*a[1, 1]/2 - u(z, mu[1])*v(z, mu[1])*a[1, 2]/2 - u(z, mu[2])*v(z, mu[2])*a[1, 1]/4 - 3*u(z, mu[2])*v(z, mu[2])*a[2, 2]/4 + a[1]/2 - a[2]/2, z))

In [81]:
(
    - _phi_dz.subs(m,2).subs(gamms).simplify().rhs 
    - _phi_dz.subs(m,1).subs(gamms).simplify().rhs
).subs(ajk_syms).factor()

(u(z, mu[1])*v(z, mu[1]) + u(z, mu[2])*v(z, mu[2]))*(a[1, 1] + 2*a[1, 2] + a[2, 2])/4

In [87]:
for _ in du_dv_hat_eqs:
    _.subs([(phi(z,1), phi(z, mu[1])), (phi(z,2), phi(z, mu[2]))]).doit().subs(_phi_mu_subs).expand()

Eq(Derivative(uhat(z, mu[1]), z), -3*u(z, mu[1])*uhat(z, mu[1])*v(z, mu[1])*a[1, 1]/4 - u(z, mu[1])*uhat(z, mu[1])*v(z, mu[1])*a[2, 2]/4 - u(z, mu[2])*uhat(z, mu[1])*v(z, mu[2])*a[1, 2]/2 + u(z, mu[2])*uhat(z, mu[1])*v(z, mu[2])*a[2, 2]/2 - uhat(z, mu[1])**2*vhat(z, mu[1])*a[1, 1] - uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[2])*a[1, 2] - 3*uhat(z, mu[1])*a[1]/2 + uhat(z, mu[1])*a[2]/2 + vhat(z, mu[2])*exp(phi(z, mu[1]))*exp(phi(z, mu[2])))

Eq(Derivative(uhat(z, mu[2]), z), u(z, mu[1])*uhat(z, mu[2])*v(z, mu[1])*a[1, 1]/2 - u(z, mu[1])*uhat(z, mu[2])*v(z, mu[1])*a[1, 2]/2 - u(z, mu[2])*uhat(z, mu[2])*v(z, mu[2])*a[1, 1]/4 - 3*u(z, mu[2])*uhat(z, mu[2])*v(z, mu[2])*a[2, 2]/4 - uhat(z, mu[1])*uhat(z, mu[2])*vhat(z, mu[1])*a[2, 1] - uhat(z, mu[2])**2*vhat(z, mu[2])*a[2, 2] + uhat(z, mu[2])*a[1]/2 - 3*uhat(z, mu[2])*a[2]/2 + vhat(z, mu[1])*exp(phi(z, mu[1]))*exp(phi(z, mu[2])))

Eq(Derivative(vhat(z, mu[1]), z), 3*u(z, mu[1])*v(z, mu[1])*vhat(z, mu[1])*a[1, 1]/4 + u(z, mu[1])*v(z, mu[1])*vhat(z, mu[1])*a[2, 2]/4 + u(z, mu[2])*v(z, mu[2])*vhat(z, mu[1])*a[1, 2]/2 - u(z, mu[2])*v(z, mu[2])*vhat(z, mu[1])*a[2, 2]/2 + uhat(z, mu[1])*vhat(z, mu[1])**2*a[1, 1] + uhat(z, mu[2])*vhat(z, mu[1])*vhat(z, mu[2])*a[1, 2] - uhat(z, mu[2])*exp(-phi(z, mu[1]))*exp(-phi(z, mu[2])) + 3*vhat(z, mu[1])*a[1]/2 - vhat(z, mu[1])*a[2]/2)

Eq(Derivative(vhat(z, mu[2]), z), -u(z, mu[1])*v(z, mu[1])*vhat(z, mu[2])*a[1, 1]/2 + u(z, mu[1])*v(z, mu[1])*vhat(z, mu[2])*a[1, 2]/2 + u(z, mu[2])*v(z, mu[2])*vhat(z, mu[2])*a[1, 1]/4 + 3*u(z, mu[2])*v(z, mu[2])*vhat(z, mu[2])*a[2, 2]/4 + uhat(z, mu[1])*vhat(z, mu[1])*vhat(z, mu[2])*a[2, 1] - uhat(z, mu[1])*exp(-phi(z, mu[1]))*exp(-phi(z, mu[2])) + uhat(z, mu[2])*vhat(z, mu[2])**2*a[2, 2] - vhat(z, mu[2])*a[1]/2 + 3*vhat(z, mu[2])*a[2]/2)